In [1]:
import os
import shutil

import torch
import transformers
from huggingface_hub import auth_check, try_to_load_from_cache
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError
from pandas import DataFrame
from transformers import AutoConfig, AutoTokenizer

from residual_stream import corpus_files, generate_missing, input_ids_with_bos, output_folder_name

GENERATE_RS = False  # residual streams are generated on the Mac; True there to (re)generate
if GENERATE_RS:
    assert torch.backends.mps.is_available(), "run this notebook on the Mac (MPS)"
print(f"transformers {transformers.__version__}, torch {torch.__version__}")

TEXT_FOLDER = 'core_clinical_short/'
# bf16 for every model: OLMo-2-0325-32B is stored in float32 (120 GiB, beyond what MPS can
# allocate) and its Instruct twin in bf16; the Gemma and Mistral checkpoints are already bf16.
DTYPE = torch.bfloat16

# Cached models first, so the downloads (4b-pt, Mistral and OLMo-2 pairs) come last.
MODELS = [
    'google/gemma-3-27b-pt',
    'google/gemma-3-27b-it',
    'google/gemma-3-12b-pt',
    'google/gemma-3-12b-it',
    'google/gemma-3-4b-pt',
    'google/gemma-3-4b-it',
    'mistralai/Mistral-Small-24B-Base-2501',
    'mistralai/Mistral-Small-24B-Instruct-2501',
    'allenai/OLMo-2-0325-32B',
    'allenai/OLMo-2-0325-32B-Instruct',
]

texts = {}
for t in corpus_files(TEXT_FOLDER):
    with open(os.path.join(TEXT_FOLDER, t), 'r', encoding='utf-8') as file:
        texts[t] = file.read()
print(f"{len(texts)} texts in {TEXT_FOLDER}")

transformers 4.57.6, torch 2.9.1
51 texts in core_clinical_short/


## Preflight (no model weights loaded)

For each model: repo access (gated repos need `huggingface-cli login` and accepted terms),
whether it is already cached, blocks/width from the config, the BOS token, whether the
tokenizer adds it itself (`native_bos`; if not, `input_ids_with_bos` prepends it, e.g.
OLMo-2), and the expected size of the full-stream output.

In [2]:
def text_config(cfg):
    # Multimodal Gemma-3 nests the language model config under text_config.
    return getattr(cfg, 'text_config', None) or cfg


rows = []
for m in MODELS:
    row = {'model': m}
    try:
        auth_check(m)
        row['access'] = 'ok'
    except GatedRepoError:
        row['access'] = 'GATED: accept terms on the hub'
    except RepositoryNotFoundError:
        row['access'] = 'NOT FOUND / not logged in'
    row['cached'] = isinstance(try_to_load_from_cache(m, 'config.json'), str)
    if row['access'] == 'ok':
        cfg = text_config(AutoConfig.from_pretrained(m, trust_remote_code=True))
        tok = AutoTokenizer.from_pretrained(m)
        ids = [input_ids_with_bos(tok, x) for x in texts.values()]
        n_tok = sum(len(i) for i in ids)
        row.update(blocks=cfg.num_hidden_layers, width=cfg.hidden_size, tokens=n_tok,
                   bos_token=tok.bos_token,
                   native_bos=all(tok(x).input_ids[0] == tok.bos_token_id for x in texts.values()),
                   out_GB=round(n_tok * (cfg.num_hidden_layers + 1) * cfg.hidden_size * 4 / 1e9, 1))
    rows.append(row)

preflight = DataFrame(rows).set_index('model')
free_GB = shutil.disk_usage('.').free / 1e9
print(f"free disk: {free_GB:.0f} GB; full-stream output total: {preflight['out_GB'].sum():.1f} GB "
      f"(plus ~50-65 GB per uncached 24-32B download)")
preflight

The tokenizer you are loading from 'mistralai/Mistral-Small-24B-Instruct-2501' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


free disk: 1674 GB; full-stream output total: 20.8 GB (plus ~50-65 GB per uncached 24-32B download)


,access,cached,blocks,width,tokens,bos_token,native_bos,out_GB
model,,,,,,,,
google/gemma-3-27b-pt,ok,True,62,5376,2216,<bos>,True,3.0
google/gemma-3-27b-it,ok,True,62,5376,2216,<bos>,True,3.0
google/gemma-3-12b-pt,ok,True,48,3840,2216,<bos>,True,1.7
google/gemma-3-12b-it,ok,True,48,3840,2216,<bos>,True,1.7
google/gemma-3-4b-pt,ok,True,34,2560,2216,<bos>,True,0.8
google/gemma-3-4b-it,ok,True,34,2560,2216,<bos>,True,0.8
mistralai/Mistral-Small-24B-Base-2501,ok,True,40,5120,2270,<s>,True,1.9
mistralai/Mistral-Small-24B-Instruct-2501,ok,True,40,5120,2270,<s>,True,1.9
allenai/OLMo-2-0325-32B,ok,True,64,5120,2253,<|endoftext|>,False,3.0


## Generate

Full stream (every layer) of `core_clinical_short` for each model with Hub access, the input
to 82's per-layer separability sweep. `generate_missing` loads one model at a time and skips
models whose folder is already complete, so no model is loaded just to find nothing left to
do. A failure is recorded and the loop moves on. 82 then picks each model's operating layer
and generates the single-layer streams there.

In [3]:
accessible = [m for m in MODELS if preflight.loc[m, 'access'] == 'ok']
print("skipped (no access):", {m: preflight.loc[m, 'access'] for m in MODELS if m not in accessible})

if GENERATE_RS:
    failed = generate_missing([(m, TEXT_FOLDER, None) for m in accessible], dtype=DTYPE)

skipped (no access): {}

===== mistralai/Mistral-Small-24B-Base-2501 / core_clinical_short/ / full stream =====


Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

41 layers with embedding layer! (torch.bfloat16 on mps:0)


Mistral-Small-24B-Base-2501:   0%|          | 0/51 [00:00<?, ?it/s]


failed: {}
